# cassandra-5-where-clause
<img src="https://upload.wikimedia.org/wikipedia/commons/5/5e/Cassandra_logo.svg" width="100" height="100">

In [1]:
#! pip install cassandra-driver

import cassandra

# Create a connection to the database.
from cassandra.cluster import Cluster
try: 
    cluster = Cluster(['127.0.0.1']) #If you have a locally installed Apache Cassandra instance
    session = cluster.connect()
except Exception as e:
    print(e)

# Create a keyspace to work in.
try:
    session.execute("""
    CREATE KEYSPACE IF NOT EXISTS udacity 
    WITH REPLICATION = 
    { 'class' : 'SimpleStrategy', 'replication_factor' : 1 }"""
)
except Exception as e:
    print(e)

# Connect to our Keyspace.
try:
    session.set_keyspace('udacity')
except Exception as e:
    print(e)


### Let's imagine we would like to start creating a new Music Library of albums. 
We want to ask 4 question of our data:
1. Give me every album in my music library that was released in a 1965 year
2. Give me the album that is in my music library that was released in 1965 by "The Beatles"
3. Give me all the albums released in a given year that was made in London 
4. Give me the city where the album "Rubber Soul" was recorded

### Here is our Collection of Data
<img src="images/cassandra-5-where-clause.jpg" width="400" height="400">

### How should we model this data? 
What should be our Primary Key and Partition Key? Since our data is looking for the YEAR let's start with that. From there we will add clustering columns on Artist Name and Album Name.

In [2]:
query = "CREATE TABLE IF NOT EXISTS music_library "
query = query + "(year int, artist_name text, album_name text, city text, PRIMARY KEY (year, artist_name, album_name))"
try:
    session.execute(query)
except Exception as e:
    print(e)

In [3]:
# Insert data.
query = "INSERT INTO music_library (year, artist_name, album_name, city)"
query = query + " VALUES (%s, %s, %s, %s)"

try:
    session.execute(query, (1970, "The Beatles", "Let it Be", "Liverpool"))
except Exception as e:
    print(e)
    
try:
    session.execute(query, (1965, "The Beatles", "Rubber Soul", "Oxford"))
except Exception as e:
    print(e)
    
try:
    session.execute(query, (1965, "The Who", "My Generation", "London"))
except Exception as e:
    print(e)

try:
    session.execute(query, (1966, "The Monkees", "The Monkees", "Los Angeles"))
except Exception as e:
    print(e)

try:
    session.execute(query, (1970, "The Carpenters", "Close To You", "San Diego"))
except Exception as e:
    print(e)

### Validate the Data Model with 4 queries.

In [4]:
# Query 1 "Give me every album in my music library that was released in a 1965 year":
query = "SELECT * FROM music_library WHERE year=1965"
try:
    rows = session.execute(query)
except Exception as e:
    print(e)
    
for row in rows:
    print (row.year, row.artist_name, row.album_name, row.city)

1965 The Beatles Rubber Soul Oxford
1965 The Who My Generation London


In [5]:
# Query 2 "Give me the album that is in my music library that was released in 1965 by "The Beatles"":
query = "SELECT * FROM music_library WHERE year=1965 AND artist_name='The Beatles'"
try:
    rows = session.execute(query)
except Exception as e:
    print(e)
    
for row in rows:
    print (row.year, row.artist_name, row.album_name, row.city)

1965 The Beatles Rubber Soul Oxford


In [6]:
# Query 3 "Give me all the albums released in a given year that was made in London":
query = "SELECT * FROM music_library WHERE year=1965 AND city='London'"
try:
    rows = session.execute(query)
except Exception as e:
    print(e)
    
for row in rows:
    print (row.year, row.artist_name, row.album_name, row.city)

Error from server: code=2200 [Invalid query] message="Cannot execute this query as it might involve data filtering and thus may have unpredictable performance. If you want to execute this query despite the performance unpredictability, use ALLOW FILTERING"


In [7]:
# Query 4 "Give me the city where the album "Rubber Soul" was recorded": 
query = "SELECT city FROM music_library WHERE year=1965 AND artist_name='The Beatles' AND album_name='Rubber Soul'"
try:
    rows = session.execute(query)
except Exception as e:
    print(e)
    
for row in rows:
    print (row.city)

Oxford


In [8]:
# Drop the table.
query = "DROP TABLE music_library"
try:
    rows = session.execute(query)
except Exception as e:
    print(e)

# Close the session and cluster connection.
session.shutdown()
cluster.shutdown()